In [ ]:
%python
# 1001 · Part B · Ingest the turbine alarms of one format
# The task's parameter file_format decides which files it reads and which table it writes.
import pyspark.sql.functions as F

dbutils.widgets.dropdown("file_format", "csv", ["csv", "json"], "File format")
file_format = dbutils.widgets.get("file_format")

SOURCE = "/Volumes/lab1001/landing/data/alarms/"
TARGET = f"lab1001.bronze.alarms_{file_format}"
print(f"Reading *.{file_format} from {SOURCE} into {TARGET}")

In [ ]:
%python
# Read only the files of this format: CSV and JSON share the folder
reader = spark.read.format(file_format).option("pathGlobFilter", f"*.{file_format}")
if file_format == "csv":
    reader = reader.option("header", "true")

alarms = reader.load(SOURCE).select(
    F.to_timestamp("event_time").alias("event_time"),
    "park_id",
    "turbine_id",
    "alarm_code",
    "severity",
    F.col("duration_min").cast("int").alias("duration_min"),
    F.col("_metadata.file_name").alias("source_file"),
    F.current_timestamp().alias("loaded_at"),
)

In [ ]:
%python
# Rebuild the table from every file in the folder: running it twice gives the same table
alarms.write.mode("overwrite").saveAsTable(TARGET)

In [ ]:
%python
# Check: every source file has the extension of this task's format
display(spark.table(TARGET).groupBy("source_file").count().orderBy("source_file"))